In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("shayanfazeli/heartbeat")

# print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import subprocess
import tempfile
import os
import re
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist

# --- Opcjonalne, w zależności od tego, jak zainstalowałeś GUDHI ---
import gudhi as gd
import gudhi.representations as gdr

def takens_embedding(signal, delay, dimension):
    """Tworzy chmurę punktów z sygnału 1D za pomocą opóźnienia czasowego."""
    n = len(signal)
    if n - (dimension - 1) * delay <= 0:
        raise ValueError("Sygnał jest za krótki dla takich parametrów osadzenia.")
    
    embedded = np.array([
        signal[i : i + dimension * delay : delay]
        for i in range(n - (dimension - 1) * delay)
    ])
    return embedded

def run_cpp_ripser(point_cloud, ripser_path="../../ripser/ripser", max_dim=1):
    """
    Liczy macierz odległości, zapisuje ją do pliku tymczasowego 
    i wywołuje plik wykonywalny C++ ripser.
    """
    # 1. Obliczenie dolnej macierzy trójkątnej (domyślny i najszybszy format dla Ripsera)
    dist_matrix = pdist(point_cloud, metric='euclidean')
    
    # Przekształcenie wektora pdist do formatu lower-distance (wartości oddzielone przecinkami)
    n_points = point_cloud.shape[0]
    
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.csv') as tmp_file:
        idx = 0
        for i in range(1, n_points):
            # Bierzemy i elementów z wektora pdist
            row_dists = dist_matrix[idx : idx + i]
            tmp_file.write(",".join(map(str, row_dists)) + "\n")
            idx += i
        tmp_filename = tmp_file.name

    # 2. Wywołanie procesu Ripser w C++
    diagrams = {}
    try:
        cmd = [ripser_path, "--format", "lower-distance", "--dim", str(max_dim), tmp_filename]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        output = result.stdout
        
        # 3. Parsowanie wyjścia (stdout) Ripsera
        current_dim = None
        for line in output.split('\n'):
            line = line.strip()
            if line.startswith("persistence intervals in dim"):
                current_dim = int(re.search(r'\d+', line).group())
                diagrams[current_dim] = []
            elif line.startswith("[") and current_dim is not None:
                # Oczyszczanie i pobieranie narodzin (birth) i śmierci (death)
                vals = line.replace("[", "").replace(")", "").split(",")
                birth = float(vals[0].strip())
                # Jeśli śmierć to nieskończoność (brak zamknięcia)
                death_str = vals[1].strip()
                death = float('inf') if death_str == '' else float(death_str)
                diagrams[current_dim].append((birth, death))
                
    finally:
        os.remove(tmp_filename) # Sprzątanie pliku tymczasowego
        
    # Konwersja list na numpy arrays (wymagane przez GUDHI)
    for dim in diagrams:
        diagrams[dim] = np.array(diagrams[dim]).reshape(-1, 2)        
    return diagrams

def main():
    # 1. Wczytanie danych (zbiór MIT-BIH jako przykład)
    # Z Kaggle wiemy, że nie ma nagłówków, a 188. kolumna to etykieta
    df = pd.read_csv("mitbih_train.csv", header=None)
    
    # Wyciągamy pierwszy sygnał klasy 0 (Normal) i klasy 1 (Supraventricular ectopic beat)
    signal_normal = df[df.iloc[:, 187] == 0.0].iloc[0, :187].values
    signal_abnormal = df[df.iloc[:, 187] == 1.0].iloc[0, :187].values

    # Parametry Takens Embedding
    delay_tau = 3
    dim_d = 4

    pc_normal = takens_embedding(signal_normal, delay_tau, dim_d)
    pc_abnormal = takens_embedding(signal_abnormal, delay_tau, dim_d)

    # 2. Obliczenia używając Twojego skompilowanego ripsera
    print("Obliczanie homologii dla sygnału normalnego...")
    diag_normal = run_cpp_ripser(pc_normal, ripser_path="../../ripser/ripser", max_dim=1)
    
    print("Obliczanie homologii dla sygnału nienormalnego...")
    diag_abnormal = run_cpp_ripser(pc_abnormal, ripser_path="../../ripser/ripser", max_dim=1)

    # Filtrujemy tylko wymiar H1 do analizy entropii i krajobrazów (pętle są najbardziej informatywne dla EKG)
    h1_normal = diag_normal.get(1, np.empty((0,2)))
    h1_abnormal = diag_abnormal.get(1, np.empty((0,2)))

    # Usuwamy nieskończoności przed wrzuceniem do uczenia maszynowego/reprezentacji
    h1_normal = h1_normal[h1_normal[:, 1] != np.inf]
    h1_abnormal = h1_abnormal[h1_abnormal[:, 1] != np.inf]

    # 3. Persistent Entropy (GUDHI)
    # Zwraca pojedynczą wartość skalarną (feature) dla danego diagramu
    entropy_calc = gdr.Entropy()
    
    ent_norm = entropy_calc.fit_transform([h1_normal])[0]
    ent_abnorm = entropy_calc.fit_transform([h1_abnormal])[0]
    
    print(f"Entropia Persystencji H1 (Normal): {ent_norm[0]:.4f}")
    print(f"Entropia Persystencji H1 (Abnormal): {ent_abnorm[0]:.4f}")

    # 4. Persistence Landscapes (GUDHI)
    # Zwraca wektor, który można łatwo podać do modelu np. Random Forest lub SVM
    landscape_calc = gdr.Landscape(num_landscapes=3, resolution=100)
    
    land_norm = landscape_calc.fit_transform([h1_normal])[0]
    land_abnorm = landscape_calc.fit_transform([h1_abnormal])[0]

    # Wizualizacja Krajobrazów
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(land_norm)
    plt.title("Persistence Landscape H1 (Normal)")
    
    plt.subplot(1, 2, 2)
    plt.plot(land_abnorm)
    plt.title("Persistence Landscape H1 (Abnormal)")
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()

Obliczanie homologii dla sygnału normalnego...
Obliczanie homologii dla sygnału nienormalnego...


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed